### Transofrm Orders Data From String to JSON

1. Pre-process the JSON string to fix the Data Quality Issues
2. Transform JSON string to JSON object
3. Write Transformed data to Silver schema

In [0]:
dfOrders = spark.table("gizmobox_sivan.bronze.v_orders")
display(dfOrders)

###1.Pre-process the JSON string to fix the Data Quality Issues
[regexp_replace function](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/regexp_replace)

In [0]:
from pyspark.sql import functions as f
dfCorrectedOrders = (
    dfOrders.select(
        "value",
        f.regexp_replace("value", '"order_date": (\\d{4}-\\d{2}-\\d{2})','"order_date": "$1"').alias("fixed_value")        
    )
)

display(dfCorrectedOrders)

###2.Transform JSON string to JSON object

[Function Schema_of_json](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/schema_of_json)

[Function from_json](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/from_json)  

In [0]:
from pyspark.sql import functions as f
dfSchemaOfOrders = (
    dfCorrectedOrders.select(
        f.schema_of_json(dfCorrectedOrders.fixed_value).alias("schema"),
        "fixed_value"
    )
)
display(dfSchemaOfOrders)

In [0]:
from pyspark.sql import functions as f
dfOrdersInJson = (
    dfSchemaOfOrders.select(
        f.from_json(
            dfSchemaOfOrders.fixed_value,
            'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>'
        ).alias("json_value")
    )
)

display(dfOrdersInJson)

###3. Write Transformed data to Silver schema

In [0]:
dfOrdersInJson.writeTo("gizmobox_sivan.silver.py_orders_json").createOrReplace()


In [0]:
df = spark.table("gizmobox_sivan.silver.py_orders_json")
display(df)
